# 02 - Routine Scraping Notebook
## Instagram Competition Monitoring & Curation System

Notebook ini dijalankan secara berkala (rekomendasi: setiap Jumat pukul 12.00 WIB) untuk mengumpulkan postingan terbaru sejak `global_checkpoint` terakhir.
- Memeriksa keberadaan `global_checkpoint` (jika kosong, eksekusi dihentikan dengan arahan menjalankan Initial Scraping terlebih dahulu sesuai D13).
- Memprioritaskan Source Accounts, kemudian keyword aktif yang jatuh tempo berdasarkan kolom `Frequency`.
- Menyimpan raw post ke System Log sebelum AI extraction.
- Melakukan matching 4 atribut terhadap Competition Report tanpa merusak formula atau data admin manual.
- Memajukan `global_checkpoint` (waktu MULAI run) saat cakupan scraping lengkap (D18), walaupun eksekusi `Partial` karena kegagalan ekstraksi, asalkan minimal satu sumber selesai di-scrape; post yang gagal diambil ulang dari System Log tanpa scraping ulang.


### 1. Setup Lingkungan & Autentikasi Google Sheets
Memuat kode modul `compmon`, menginstal dependensi dari `requirements.txt`, dan melakukan autentikasi via `google.auth.default()` + `gspread.authorize()`.

In [ ]:
# @title 1. Muat Kode & Instal Dependensi Terpin
import os, sys
sys.path.append(os.getcwd())

!pip install -q -r requirements.txt

from google.colab import auth, userdata
import google.auth
import gspread

# Autentikasi akun Google resmi Colab (Instruction 13)
auth.authenticate_user()
creds, _ = google.auth.default()
gc = gspread.authorize(creds)

# Membaca kredensial dari Colab Secrets
SPREADSHEET_ID = userdata.get('SPREADSHEET_ID')
APIFY_API_TOKEN = userdata.get('APIFY_API_TOKEN')
GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')

### 2. Eksekusi Routine Scraping & Pemrosesan AI

In [ ]:
from compmon.sheets_repo import SheetsRepository
from compmon.pipeline import CompetitionPipeline

repo = SheetsRepository(gc, SPREADSHEET_ID)
pipeline = CompetitionPipeline(
    sheets_repo=repo,
    apify_token=APIFY_API_TOKEN,
    gemini_api_key=GEMINI_API_KEY,
    notebook_name="ROUTINE"
)

summary = pipeline.run_routine_scraping()

print("=== Ringkasan Eksekusi Routine Scraping ===")
for key, val in summary.items():
    print(f"{key}: {val}")